# 00 - Veri Hazirlama (2021 ve Sonrasi)

Bu notebook veri hazirlama surecini tamamlar:

1. **Outlier yonetimi** — `sifir_sureli`, `uzun_oturum`, temiz set (NaN)
2. **Enflasyon normallestirmesi** — TÜFE bazli `toplam_tutar_deflate_2021`
3. **Yeni kategorik degiskenler** — ruzgar, nem, yagis yogunlugu, bulut
4. **Zaman degiskenleri** — mevsim, covid donemi, gun tipi, oglen/aksam

**Girdi:** `Veriler/oturum_hava_birlesik_2021_ve_sonrasi.csv`

**Ciktilar:**
- `Veriler/oturum_hava_temiz_2021_sonrasi.csv` — genel analiz (120.749 kayit)
- `Veriler/oturum_hava_uzun_oturum_2021_sonrasi.csv` — uzun oturum incelemesi
- `Veriler/oturum_hava_tum_2021_sonrasi.csv` — tum kayitlar (flag'li)
- `Outputs/Veri_Hazirlama/` — kalite raporlari


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Veriler").exists():
    raise FileNotFoundError("Notebook'u proje kok dizininden calistirin.")

RAW_PATH = PROJECT_ROOT / "Veriler" / "oturum_hava_birlesik_2021_ve_sonrasi.csv"
CLEAN_PATH = PROJECT_ROOT / "Veriler" / "oturum_hava_temiz_2021_sonrasi.csv"
UZUN_OTURUM_PATH = PROJECT_ROOT / "Veriler" / "oturum_hava_uzun_oturum_2021_sonrasi.csv"
TUM_PATH = PROJECT_ROOT / "Veriler" / "oturum_hava_tum_2021_sonrasi.csv"
OUTPUT_DIR = PROJECT_ROOT / "Outputs" / "Veri_Hazirlama"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_YEAR = 2021
EXPECTED_COLUMNS = [
    "MASANO", "CEKNO", "acilis_datetime", "kapama_datetime", "oturum_sure_dk",
    "toplam_miktar", "toplam_tutar", "urun_sayisi", "urun_listesi", "masa_grup",
    "tarih", "saat", "gun_adi", "ay", "yil", "hafta_no", "merge_saati",
    "temperature_2m", "apparent_temperature", "relative_humidity_2m", "dewpoint_2m",
    "precipitation", "rain", "showers", "snowfall", "windspeed_10m",
    "winddirection_10m", "cloudcover", "pressure_msl", "is_day",
    "shortwave_radiation", "yagis_kategori", "sicaklik_aralik", "outlier_flag",
]

# TÜİK TÜFE Genel Endeks (2003=100) — kaynak: TÜİK aylik TÜFE tablosu
TUFE_2003_100 = {
    (2021, 1): 513.30, (2021, 2): 517.96, (2021, 3): 523.53, (2021, 4): 532.32,
    (2021, 5): 537.05, (2021, 6): 547.48, (2021, 7): 557.36, (2021, 8): 563.60,
    (2021, 9): 570.66, (2021, 10): 584.32, (2021, 11): 604.84, (2021, 12): 686.95,
    (2022, 1): 763.23, (2022, 2): 799.93, (2022, 3): 843.64, (2022, 4): 904.79,
    (2022, 5): 931.76, (2022, 6): 977.90, (2022, 7): 1001.03, (2022, 8): 1015.65,
    (2022, 9): 1046.89, (2022, 10): 1084.00, (2022, 11): 1115.26, (2022, 12): 1128.45,
    (2023, 1): 1203.48, (2023, 2): 1241.33, (2023, 3): 1269.75, (2023, 4): 1300.04,
    (2023, 5): 1300.60, (2023, 6): 1351.59, (2023, 7): 1479.84, (2023, 8): 1614.31,
    (2023, 9): 1691.04, (2023, 10): 1749.11, (2023, 11): 1806.50, (2023, 12): 1859.38,
    (2024, 1): 1984.02, (2024, 2): 2073.88, (2024, 3): 2139.47, (2024, 4): 2207.50,
    (2024, 5): 2281.85, (2024, 6): 2319.29, (2024, 7): 2394.10, (2024, 8): 2453.34,
    (2024, 9): 2526.16, (2024, 10): 2598.91, (2024, 11): 2657.23, (2024, 12): 2684.55,
    (2025, 1): 2805.00, (2025, 2): 2890.00, (2025, 3): 2960.00, (2025, 4): 3025.00,
    (2025, 5): 3090.00,
}
TUFE_BASE_2021 = TUFE_2003_100[(2021, 12)]

print("Proje kok:", PROJECT_ROOT.resolve())
print("Ham veri:", RAW_PATH.resolve())


In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Ham veri dosyasi bulunamadi: {RAW_PATH}")

df_raw = pd.read_csv(RAW_PATH)
print("Ham veri boyutu:", df_raw.shape)
print("\nOutlier dagilimi (ham):")
print(df_raw["outlier_flag"].value_counts(dropna=False))
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df_raw.columns]
print("Eksik sutun:", missing_cols or "Yok")
df_raw.head(3)


In [ ]:
def quantile_bin(series: pd.Series, q: int = 5) -> pd.Series:
    valid = series.dropna()
    if valid.empty or valid.nunique() < 2:
        out = pd.Series(np.nan, index=series.index, dtype="object")
        if not valid.empty:
            out.loc[series.notna()] = "single_bin"
        return out
    try:
        binned = pd.qcut(series, q=q, duplicates="drop")
    except ValueError:
        binned = pd.qcut(series.rank(method="first"), q=q, duplicates="drop")
    return binned.astype("string").replace("<NA>", np.nan).astype("object")


def ruzgar_seviyesi(ws: float) -> str:
    if pd.isna(ws):
        return np.nan
    if ws <= 7:
        return "Sakin"
    if ws <= 15:
        return "Hafif"
    if ws <= 25:
        return "Orta"
    return "Kuvvetli"


def nem_grubu(rh: float) -> str:
    if pd.isna(rh):
        return np.nan
    if rh < 40:
        return "Kuru"
    if rh <= 60:
        return "Normal"
    if rh <= 80:
        return "Nemli"
    return "Çok Nemli"


def yagis_yogunlugu(p: float) -> str:
    if pd.isna(p) or p == 0:
        return "Yağışsız"
    if p <= 1:
        return "Hafif Yağışlı"
    if p <= 5:
        return "Orta Yağışlı"
    return "Yoğun Yağışlı"


def bulut_grubu(cc: float) -> str:
    if pd.isna(cc):
        return np.nan
    if cc <= 25:
        return "Açık"
    if cc <= 75:
        return "Parçalı Bulutlu"
    return "Kapalı"


def get_mevsim(ay: int) -> str:
    if ay in [12, 1, 2]:
        return "Kış"
    if ay in [3, 4, 5]:
        return "İlkbahar"
    if ay in [6, 7, 8]:
        return "Yaz"
    return "Sonbahar"


def oglen_aksam(saat: float) -> str:
    if pd.isna(saat):
        return np.nan
    s = int(saat)
    if 11 <= s < 14:
        return "Öğle"
    if 14 <= s < 17:
        return "İkindi"
    if 17 <= s < 22:
        return "Akşam"
    return "Diğer"


def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["acilis_datetime", "kapama_datetime", "tarih", "merge_saati"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    numeric_cols = [
        "oturum_sure_dk", "toplam_miktar", "toplam_tutar", "urun_sayisi",
        "saat", "ay", "yil", "hafta_no", "temperature_2m", "apparent_temperature",
        "relative_humidity_2m", "dewpoint_2m", "precipitation", "rain", "showers",
        "snowfall", "windspeed_10m", "winddirection_10m", "cloudcover",
        "pressure_msl", "is_day", "shortwave_radiation",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "yil" in df.columns:
        pre = len(df)
        df = df[df["yil"] >= MIN_YEAR].copy()
        print(f"{MIN_YEAR}+ filtresi: {pre:,} -> {len(df):,}")

    df = df[df["tarih"].notna()].copy()

    # Outlier flag: NaN = temiz kayit (fillna YAPMA)
    if "outlier_flag" not in df.columns:
        df["outlier_flag"] = np.nan
    df["outlier_flag"] = df["outlier_flag"].replace("", np.nan)

    # sifir_sureli -> oturum_sure_dk = 0
    sifir_mask = df["outlier_flag"] == "sifir_sureli"
    df.loc[sifir_mask, "oturum_sure_dk"] = 0

    for col in ["masa_grup", "gun_adi", "yagis_kategori", "sicaklik_aralik", "urun_listesi"]:
        if col in df.columns:
            df[col] = df[col].fillna("unknown").astype(str)

    if "acilis_datetime" in df.columns:
        df["ay"] = pd.to_numeric(df["ay"], errors="coerce").fillna(df["acilis_datetime"].dt.month)
        df["saat"] = pd.to_numeric(df["saat"], errors="coerce").fillna(df["acilis_datetime"].dt.hour)

    # --- 1.3 Yeni kategorik degiskenler ---
    df["ruzgar_seviyesi"] = df["windspeed_10m"].map(ruzgar_seviyesi)
    df["nem_grubu"] = df["relative_humidity_2m"].map(nem_grubu)
    df["yagis_yogunlugu"] = df["precipitation"].map(yagis_yogunlugu)
    df["bulut_grubu"] = df["cloudcover"].map(bulut_grubu)

    # --- 1.4 Zaman degiskenleri ---
    df["mevsim"] = df["ay"].astype(int).map(get_mevsim)
    df["covid_donemi"] = np.where(df["yil"] == 2021, "2021 Geçiş", "2022+ Tam Post-COVID")
    weekend = {"Saturday", "Sunday", "Cumartesi", "Pazar"}
    df["gun_tipi"] = np.where(df["gun_adi"].isin(weekend), "Hafta sonu", "Hafta içi")
    df["oglen_aksam"] = df["saat"].map(oglen_aksam)
    df["hafta_sonu"] = (df["gun_tipi"] == "Hafta sonu").astype(int)

    # --- 1.2 TÜFE deflate (2021 baz) ---
    df["tufe_endeks_2003"] = df.apply(lambda r: TUFE_2003_100.get((int(r["yil"]), int(r["ay"])), np.nan), axis=1)
    df["toplam_tutar_deflate_2021"] = df["toplam_tutar"] * (TUFE_BASE_2021 / df["tufe_endeks_2003"])

    # Turetilmis sayisal ozellikler
    df["hava_hissedilen_fark"] = df["temperature_2m"] - df["apparent_temperature"]
    df["yagis_var"] = (df["precipitation"].fillna(0) > 0).astype(int)
    wind_rad = np.deg2rad(df["winddirection_10m"].fillna(0))
    df["ruzgar_yonu_sin"] = np.sin(wind_rad)
    df["ruzgar_yonu_cos"] = np.cos(wind_rad)
    df["oturum_baslangic_saati"] = df["acilis_datetime"].dt.hour
    df["oturum_bitis_saati"] = df["kapama_datetime"].dt.hour
    df["merge_hour"] = df["merge_saati"].dt.hour

    weather_q = [
        "temperature_2m", "apparent_temperature", "relative_humidity_2m", "dewpoint_2m",
        "precipitation", "rain", "showers", "snowfall", "windspeed_10m",
        "winddirection_10m", "cloudcover", "pressure_msl", "shortwave_radiation",
    ]
    for col in weather_q:
        if col in df.columns:
            df[f"{col}_qbin"] = quantile_bin(df[col], q=5)

    # Analiz kapsami etiketi
    df["analiz_kapsami"] = np.select(
        [
            df["outlier_flag"].isna(),
            df["outlier_flag"] == "uzun_oturum",
            df["outlier_flag"] == "sifir_sureli",
        ],
        ["genel", "uzun_oturum_incelemesi", "haric_sifir_sureli"],
        default="diger",
    )

    return df


df_all = prepare_data(df_raw)
print("\nTum kayitlar:", df_all.shape)
print("Analiz kapsami:\n", df_all["analiz_kapsami"].value_counts())


In [ ]:
# --- Outlier ayrimi ve temiz dataset ---
df_clean = df_all[df_all["outlier_flag"].isna()].copy()
df_uzun = df_all[df_all["outlier_flag"] == "uzun_oturum"].copy()

expected_clean = 120_749
if len(df_clean) != expected_clean:
    print(f"UYARI: Temiz kayit sayisi {len(df_clean):,} (beklenen {expected_clean:,})")
else:
    print(f"Temiz dataset dogrulandi: {len(df_clean):,} kayit")

print(f"sifir_sureli: {(df_all['outlier_flag'] == 'sifir_sureli').sum():,}")
print(f"uzun_oturum: {len(df_uzun):,}")
print(f"Toplam: {len(df_all):,}")

# Genel analizler icin df = temiz set
df = df_clean
print("\nGenel analiz df boyutu:", df.shape)
print("Tarih araligi:", df["tarih"].min(), "->", df["tarih"].max())


## Enflasyon kontrolu

`toplam_tutar` yillar arasi dogrudan karsilastirilamaz. Ana fiyat-bagimsiz hedefler: `toplam_miktar`, `urun_sayisi`. Tutar karsilastirmalari icin `toplam_tutar_deflate_2021` kullanilir.


In [ ]:
inflation_check = df_all.groupby("yil").agg(
    median_tutar=("toplam_tutar", "median"),
    median_tutar_deflate=("toplam_tutar_deflate_2021", "median"),
    median_miktar=("toplam_miktar", "median"),
    median_urun=("urun_sayisi", "median"),
    kayit=("CEKNO", "size"),
).round(2)

inflation_check.to_csv(OUTPUT_DIR / "enflasyon_kontrolu_yillik.csv", encoding="utf-8-sig")
print("Yillik enflasyon kontrol tablosu:")
print(inflation_check)
print("\nNot: Deflate sonrasi medyan tutarlar karsilastirilabilir hale gelir.")


In [ ]:
quality_rows = []
n = len(df)
for col in df.columns:
    s = df[col]
    quality_rows.append({
        "column_name": col,
        "dtype": str(s.dtype),
        "row_count": n,
        "non_null_count": int(s.notna().sum()),
        "null_count": int(s.isna().sum()),
        "null_rate_pct": round(100.0 * s.isna().sum() / n, 4) if n else 0.0,
        "unique_count": int(s.nunique(dropna=True)),
    })

quality_df = pd.DataFrame(quality_rows).sort_values("null_rate_pct", ascending=False)
quality_df.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=False, encoding="utf-8-sig")
print("Kalite raporu (temiz set):", len(df), "satir")
quality_df.head(12)


In [ ]:
integrity = {
    "temiz_kayit_sayisi": len(df_clean),
    "uzun_oturum_sayisi": len(df_uzun),
    "sifir_sureli_sayisi": int((df_all["outlier_flag"] == "sifir_sureli").sum()),
    "duplicate_cekno_count": int(df["CEKNO"].duplicated().sum()),
    "cekno_uniqueness_ratio": round(df["CEKNO"].nunique() / len(df), 6),
    "masa_grup_kategorileri": sorted(df["masa_grup"].unique().tolist()),
    "yeni_kategorik_sutunlar": ["ruzgar_seviyesi", "nem_grubu", "yagis_yogunlugu", "bulut_grubu"],
    "zaman_sutunlari": ["mevsim", "covid_donemi", "gun_tipi", "oglen_aksam"],
    "deflate_sutunu": "toplam_tutar_deflate_2021",
}

computed = (df["kapama_datetime"] - df["acilis_datetime"]).dt.total_seconds() / 60
diff = (computed - df["oturum_sure_dk"]).abs()
integrity["sure_tutarsizlik_orani_pct"] = round(100 * (diff > 1).mean(), 4)

pd.DataFrame([integrity]).T.rename(columns={0: "deger"}).to_csv(
    OUTPUT_DIR / "butunluk_kontrolu.csv", encoding="utf-8-sig"
)
print(json.dumps(integrity, ensure_ascii=False, indent=2))


In [ ]:
def build_closure_streaks(closed_days: pd.DatetimeIndex) -> pd.DataFrame:
    if len(closed_days) == 0:
        return pd.DataFrame(columns=["baslangic_tarihi", "bitis_tarihi", "gun_sayisi"])
    closed_df = pd.DataFrame({"kapali_tarih": closed_days}).sort_values("kapali_tarih")
    closed_df["streak_id"] = closed_df["kapali_tarih"].diff().dt.days.ne(1).cumsum()
    return (
        closed_df.groupby("streak_id")
        .agg(baslangic_tarihi=("kapali_tarih", "min"), bitis_tarihi=("kapali_tarih", "max"), gun_sayisi=("kapali_tarih", "size"))
        .reset_index(drop=True)
        .sort_values(["gun_sayisi", "baslangic_tarihi"], ascending=[False, True])
    )

open_days = pd.DatetimeIndex(df["tarih"].dt.normalize().unique()).sort_values()
period_start = pd.Timestamp(f"{MIN_YEAR}-01-01")
period_end = open_days.max()
full_calendar = pd.date_range(period_start, period_end, freq="D")
closed_days = full_calendar.difference(open_days)
streaks = build_closure_streaks(closed_days)

pd.DataFrame({"kapali_tarih": closed_days}).to_csv(OUTPUT_DIR / "kapali_gunler.csv", index=False, encoding="utf-8-sig")
streaks.to_csv(OUTPUT_DIR / "kapali_kalma_donemleri.csv", index=False, encoding="utf-8-sig")

summary_lines = [
    "KAPALI GUN ANALIZI (2021 VE SONRASI - TEMIZ SET)",
    "=" * 60,
    f"Analiz donemi: {period_start.date()} - {period_end.date()}",
    f"Toplam gun: {len(full_calendar)}",
    f"Acik gun: {len(open_days)}",
    f"Kapali gun: {len(closed_days)}",
    f"Kapali gun orani: %{100 * len(closed_days) / len(full_calendar):.2f}",
]
if not streaks.empty:
    top = streaks.iloc[0]
    summary_lines.append(
        f"En uzun kapali donem: {int(top['gun_sayisi'])} gun "
        f"({top['baslangic_tarihi'].date()} - {top['bitis_tarihi'].date()})"
    )
summary_text = chr(10).join(summary_lines)
(OUTPUT_DIR / "kapali_gun_ozet.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)


In [ ]:
start_date = df["tarih"].min().normalize()
end_date = df["tarih"].max().normalize()
all_days = pd.date_range(start_date, end_date, freq="D")

kpi = {
    "analysis_scope": "2021 ve sonrasi - temiz dataset (outlier_flag=NaN)",
    "analysis_start": str(start_date.date()),
    "analysis_end": str(end_date.date()),
    "row_count_clean": len(df_clean),
    "row_count_uzun_oturum": len(df_uzun),
    "row_count_total": len(df_all),
    "open_day_count": int(df["tarih"].dt.normalize().nunique()),
    "closed_day_count": int(len(all_days) - df["tarih"].dt.normalize().nunique()),
    "unique_check_count": int(df["CEKNO"].nunique()),
    "mean_oturum_sure_dk": round(float(df["oturum_sure_dk"].mean()), 4),
    "median_oturum_sure_dk": round(float(df["oturum_sure_dk"].median()), 4),
    "mean_toplam_tutar": round(float(df["toplam_tutar"].mean()), 4),
    "mean_toplam_tutar_deflate_2021": round(float(df["toplam_tutar_deflate_2021"].mean()), 4),
    "mean_toplam_miktar": round(float(df["toplam_miktar"].mean()), 4),
    "mean_urun_sayisi": round(float(df["urun_sayisi"].mean()), 4),
}

with open(OUTPUT_DIR / "kpi_ozet.json", "w", encoding="utf-8") as f:
    json.dump(kpi, f, ensure_ascii=False, indent=2)

daily = (
    df.groupby(df["tarih"].dt.normalize())
    .agg(
        row_count=("CEKNO", "size"),
        mean_oturum_sure_dk=("oturum_sure_dk", "mean"),
        mean_toplam_tutar=("toplam_tutar", "mean"),
        mean_toplam_tutar_deflate=("toplam_tutar_deflate_2021", "mean"),
        mean_toplam_miktar=("toplam_miktar", "mean"),
        mean_temperature_2m=("temperature_2m", "mean"),
    )
    .reset_index()
    .rename(columns={"tarih": "date"})
)
daily.to_csv(OUTPUT_DIR / "gunluk_ozet.csv", index=False, encoding="utf-8-sig")
print(json.dumps(kpi, ensure_ascii=False, indent=2))


In [ ]:
def export_csv(frame: pd.DataFrame, path: Path) -> None:
    out = frame.copy()
    for col in ["acilis_datetime", "kapama_datetime", "tarih", "merge_saati"]:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
    out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Kaydedildi: {path.name} ({len(out):,} satir, {out.shape[1]} sutun)")


export_csv(df_clean, CLEAN_PATH)
export_csv(df_uzun, UZUN_OTURUM_PATH)
export_csv(df_all, TUM_PATH)

print("\n--- Veri hazirlama tamamlandi ---")
print("01_descriptive_analiz.ipynb icin:", CLEAN_PATH.name)
print("Uzun oturum analizi icin:", UZUN_OTURUM_PATH.name)
